# Global Oil & Gas Industry Analysis (1932–2014)
### Data Cleaning, Exploratory Data Analysis and Insights

**Project objective:** Explore historical country-level oil and natural gas production, prices, economic values, exports and population indicators from 1932 to 2014.

**Dataset:** `Oil and Gas 1932-2014.csv`  
**Scope:** Country-year observations. This is not field-level Abu Dhabi operational data.

## Notebook workflow
1. Import libraries and load data
2. Inspect data structure and quality
3. Clean and validate records
4. EDA and univariate analysis
5. Bivariate and multivariate analysis
6. Statistical analysis
7. Visualizations and insight generation
8. Feature engineering
9. Findings, limitations and recommendations

**Data-quality principle:** Missing values are not automatically converted to zero. Extreme values are flagged for investigation, not automatically removed, because large production and export values may be genuine.

## 1. Import libraries

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
sns.set_theme(style='whitegrid')

ModuleNotFoundError: No module named 'pandas'

## 2. Load the original dataset

In [ ]:
# Place the original CSV in the same folder as this notebook.
file_path = Path('Oil and Gas 1932-2014.csv')
if not file_path.exists():
    file_path = Path('Oil_and_Gas_1932_2014_Cleaned.csv')

df_raw = pd.read_csv(file_path)
df = df_raw.copy()

print('Shape:', df.shape)
display(df.head())

## 3. Initial data inspection

In [ ]:
display(df.info())
display(df.describe(include='all').T)
print('Duplicate rows:', df.duplicated().sum())
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'missing_count': missing, 'missing_percent': missing_pct})
display(missing_report)

## 4. Data cleaning

### Cleaning decisions
- Remove exact duplicate rows.
- Strip leading/trailing whitespace from text fields.
- Convert blank strings to missing values.
- Convert year, numeric country code and sovereignty flag to nullable integer types.
- Retain relevant columns; the supplied fields are relevant to the stated project.
- Preserve missing observations instead of blindly imputing zero or mean values.
- Check the year range against the dataset period (1932–2014).
- Detect potential outliers with the IQR rule; do not automatically delete valid country-scale extremes.

In [ ]:
rows_before = len(df)
df = df.replace(r'^\\s*$', pd.NA, regex=True)

for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].astype('string').str.strip()

df = df.drop_duplicates().reset_index(drop=True)

for col in ['year', 'iso3numeric', 'sovereign']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

if 'year' in df.columns:
    invalid_year = ~df['year'].between(1932, 2014)
    print('Invalid/out-of-period year values:', int(invalid_year.sum()))
    df.loc[invalid_year, 'year'] = pd.NA

print('Rows before:', rows_before)
print('Rows after:', len(df))
print('Exact duplicates removed:', rows_before-len(df))
display(df.head())

### 4.1 Validate text and numeric columns

In [ ]:
text_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()
numeric_cols = df.select_dtypes(include='number').columns.tolist()
print('Text columns:', text_cols)
print('Numeric columns:', numeric_cols)
print('Year range:', df['year'].min(), 'to', df['year'].max())
print('Country count:', df['cty_name'].nunique())
print('Country-year duplicate keys:', df.duplicated(['cty_name', 'year']).sum())

### 4.2 Missing-value report

In [6]:
missing_report = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percent': (df.isna().mean()*100).round(2)
}).sort_values('missing_percent', ascending=False)
display(missing_report)
plt.figure(figsize=(12, 6))
missing_report['missing_percent'].sort_values().plot(kind='barh')
plt.title('Missing values by column (%)')
plt.xlabel('Missing values (%)')
plt.tight_layout()
plt.show()

NameError: name 'pd' is not defined

### 4.3 Outlier detection (IQR)

In [ ]:
# IQR flags are diagnostic only. Do not drop records automatically.
outlier_rows = []
for col in numeric_cols:
    s = df[col].dropna()
    if s.empty or s.nunique() < 4:
        continue
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3-q1
    lower, upper = q1-1.5*iqr, q3+1.5*iqr
    count = ((s < lower) | (s > upper)).sum()
    outlier_rows.append([col, lower, upper, int(count), round(count/len(s)*100,2)])
outlier_report = pd.DataFrame(outlier_rows, columns=['column','lower_bound','upper_bound','outlier_count','outlier_percent'])
display(outlier_report.sort_values('outlier_count', ascending=False))

### 4.4 Save cleaned dataset

In [ ]:
cleaned_file = 'Oil_and_Gas_1932_2014_Cleaned.csv'
df.to_csv(cleaned_file, index=False)
print('Saved:', cleaned_file)
print('Final shape:', df.shape)

## 5. Exploratory Data Analysis (EDA)

In [ ]:
print('Countries:', df['cty_name'].nunique())
print('Years:', df['year'].min(), '-', df['year'].max())
print('Observations:', len(df))
display(df.groupby('year').size().describe())
display(df.groupby('cty_name').size().sort_values(ascending=False).head(10))

## 6. Univariate analysis

In [ ]:
# Distribution summaries for selected measures
candidate_cols = ['oil_prod32_14', 'gas_prod55_14', 'oil_price_2000',
                  'oil_price_nom', 'oil_exports', 'gas_exports', 'population']
selected = [c for c in candidate_cols if c in df.columns]
display(df[selected].describe().T)
for col in selected:
    values = df[col].dropna()
    if values.empty:
        continue
    plt.figure(figsize=(8, 4))
    sns.histplot(values, kde=True)
    plt.title(f'Distribution: {col}')
    plt.tight_layout()
    plt.show()

## 7. Bivariate analysis

In [ ]:
# Oil production and oil price relationship
if {'oil_prod32_14','oil_price_2000'}.issubset(df.columns):
    sample = df[['oil_prod32_14','oil_price_2000']].dropna()
    if len(sample):
        sns.scatterplot(data=sample, x='oil_price_2000', y='oil_prod32_14', alpha=.35)
        plt.title('Oil production vs oil price (2000 USD)')
        plt.tight_layout()
        plt.show()
        print('Pearson correlation:', sample.corr().iloc[0,1])

## 8. Multivariate analysis

In [ ]:
# Country-level aggregate comparison of production indicators
prod_cols = [c for c in ['oil_prod32_14','gas_prod55_14'] if c in df.columns]
if prod_cols:
    country_prod = df.groupby('cty_name')[prod_cols].sum(min_count=1)
    display(country_prod.sort_values(prod_cols[0], ascending=False).head(15))
    plt.figure(figsize=(10, 6))
    country_prod.sort_values(prod_cols[0], ascending=False).head(10)[prod_cols[0]].sort_values().plot(kind='barh')
    plt.title('Top 10 countries by cumulative oil production measure')
    plt.tight_layout()
    plt.show()

## 9. Statistical analysis

In [ ]:
numeric_df = df.select_dtypes(include='number')
stats = numeric_df.agg(['count','mean','median','std','min','max']).T
display(stats)
# Mode is reported separately because a column can have multiple modes.
mode_report = {col: df[col].mode(dropna=True).head(3).tolist() for col in numeric_df.columns}
display(pd.Series(mode_report, name='mode_values').to_frame())

In [ ]:
corr = numeric_df.corr(method='pearson')
plt.figure(figsize=(16, 12))
sns.heatmap(corr, cmap='coolwarm', center=0, linewidths=.2)
plt.title('Correlation matrix of numeric variables')
plt.tight_layout()
plt.show()

## 10. Data visualization – time trends

In [ ]:
annual = df.groupby('year').agg(
    oil_production=('oil_prod32_14','sum'),
    gas_production=('gas_prod55_14','sum'),
    oil_value=('oil_value_2000','sum'),
    gas_value=('gas_value_2000','sum')
)
display(annual.head())
for col in ['oil_production','gas_production']:
    plt.figure(figsize=(12,4))
    annual[col].plot()
    plt.title(f'Global total by year: {col.replace("_", " ").title()}')
    plt.xlabel('Year')
    plt.ylabel('Sum of available country values')
    plt.tight_layout()
    plt.show()

## 11. Feature engineering

In [ ]:
# Derived calendar and analytical features; original variables remain unchanged.
df['decade'] = (df['year'] // 10) * 10
df['oil_gas_production_total'] = df[['oil_prod32_14','gas_prod55_14']].sum(axis=1, min_count=1)
df['oil_gas_value_per_capita_2000'] = df['oil_gas_valuePOP_2000']
df['has_oil_production_record'] = df['oil_prod32_14'].notna().astype('int8')
df['has_gas_production_record'] = df['gas_prod55_14'].notna().astype('int8')
display(df[['cty_name','year','decade','oil_gas_production_total',
            'has_oil_production_record','has_gas_production_record']].head())

## 12. Insights and reporting

In [7]:
# Produce data-driven summary tables for the report.
if 'oil_prod32_14' in df:
    top_oil = df.groupby('cty_name')['oil_prod32_14'].sum(min_count=1).sort_values(ascending=False).head(10)
    print('Top countries by cumulative oil production measure:')
    display(top_oil.to_frame('oil_prod32_14_sum'))
if 'gas_prod55_14' in df:
    top_gas = df.groupby('cty_name')['gas_prod55_14'].sum(min_count=1).sort_values(ascending=False).head(10)
    print('Top countries by cumulative gas production measure:')
    display(top_gas.to_frame('gas_prod55_14_sum'))
print('Coverage by decade:')
display(df.groupby('decade').size().to_frame('observations'))

NameError: name 'df' is not defined

## 13. Conclusions and future recommendations

Use the generated tables and charts to write conclusions supported by the actual results. Investigate:
- Which countries account for the largest cumulative oil and gas production measures?
- How did production and economic value change across decades?
- Which country-year observations have unusually high values, and are they valid?
- Are oil/gas production, exports, population and economic-value measures correlated?
- Where is data coverage weak because of missing observations?

### Recommended next steps
1. Validate definitions, units and methodology against the dataset documentation/source.
2. Investigate high-missingness export variables before drawing country comparisons.
3. Compare results using consistent country and year coverage.
4. Review flagged outliers against source records rather than deleting them automatically.
5. Consider time-series forecasting only after checking stationarity, coverage and structural breaks.
6. Keep country-level findings separate from field-level or Abu Dhabi-specific operational conclusions.

## 14. Limitations
- This dataset is country-level historical data, not individual oil-field production data.
- Missing values may reflect unavailable or unreported observations.
- The meaning and units of abbreviated source columns should be confirmed from the original dataset documentation before external publication.
- Correlation does not establish causation.